# Male Loneliness Project — Reddit data: pull → combine → clean → tag → sample

**Run the cells top to bottom. Start with `TEST_MODE = True`** (settings cell): a few minutes, tiny data, files get a `_TEST` suffix. When it works, set `TEST_MODE = False` and run again for real.

| Step | What it does | Main output (in `project_outputs/`) |
|---|---|---|
| A | Re-pulls **only the incomplete months** of r/lonely and r/ForeverAlone (found in your old Diagnostics file) | `checkpoints/repull_*.csv` |
| B | r/AskMen **posts** with the new keyword list | `checkpoints/askmen_posts.csv` |
| C | r/exredpill **posts** (own cell) | `checkpoints/exredpill_posts.csv` |
| D | r/AskMen **comments** (top-level replies) on 50 random keyword-matched threads per year | `checkpoints/askmen_comments.csv` |
| E | Combines old file (r/AskMen and r/MGTOW removed) + A–D | `combined_raw.csv`, `old_1B_without_AskMen_MGTOW.csv` |
| F | Cleans and tags (pandas); prints a cleaning log and an F-tag report | `cleaned_all.csv`, `cleaning_log.csv`, `f_tag_report.csv` |
| G | Draws an equal random sample per community for the topic model | `analysis_sample.csv`, `analysis_sample_no_F_tags.csv` |

**Safe to stop and restart.** Pull cells save to disk as they go (every ~1,000 records) and remember which weeks/threads are finished. If a cell stops or you close the laptop, just run the same cell again and it continues. Steps A, B and D can each take hours (depends on how busy the free archive is) — run them overnight.

**Before you run:** (1) put the **same `AUTHOR_SALT`** you used in the old notebook in the settings cell; (2) check `OLD_POSTS_CSV` and `OLD_DIAG_CSV` point to your old files (use full paths if they are in Downloads); (3) the old files stay untouched.

## Part 0 — Setup

In [1]:
# Run once per environment (safe to re-run). %pip installs into the kernel you're actually using.
%pip install -q requests pandas numpy langdetect


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# SETTINGS  (run this first; every later cell uses these)
# ============================================================
import os, re, time, json, html, hashlib, warnings
import requests
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------- SAFETY SWITCH ----------
# True  = tiny test run (last few weeks, 3 threads) -> checks everything works in minutes.
# False = the real run (hours). Always do a True run first.
TEST_MODE = False
TEST_WEEKS = 4

# ---------- study window ----------
START_DATE = "2020-01-01"
END_DATE = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d")     # always "today"

# ---------- folders and files ----------
OUTPUT_DIR = "C:/Users/vesta/Downloads/project_outputs"            # same folder as your earlier outputs
# If your old files are somewhere else, use the full path, e.g.
# "C:/Users/vesta/Downloads/project_outputs/reddit_1B_posts_by_popularity__All_matching_posts.csv"
OLD_POSTS_CSV = os.path.join(OUTPUT_DIR, "reddit_1B_posts_by_popularity__All_matching_posts.csv")
OLD_DIAG_CSV  = os.path.join(OUTPUT_DIR, "reddit_1B_posts_by_popularity__Diagnostics.csv")

TAG = "_TEST" if TEST_MODE else ""        # test files get "_TEST" so they never mix with real ones
CHECK_DIR = os.path.join(OUTPUT_DIR, "checkpoints" + TAG)
os.makedirs(CHECK_DIR, exist_ok=True)

# ---------- communities ----------
SUBREDDITS_OLD = ["lonely", "ForeverAlone"]     # already pulled earlier; only their incomplete months are re-pulled
DROP_FROM_OLD  = ["AskMen", "MGTOW"]            # removed from the old file (AskMen = partial run; MGTOW = banned)
ASKMEN    = "AskMen"
EXREDPILL = "exredpill"

# ---------- r/AskMen comments ----------
THREADS_PER_YEAR     = 100     # random keyword-matched AskMen posts per year whose comments we collect
MIN_COMMENTS_IN_THREAD = 20   # only threads with at least this many comments
MAX_RAW_COMMENTS     = 2000   # safety cap on comments downloaded per thread
MAX_TOPLEVEL         = 500    # keep at most this many top-level comments per thread

# ---------- cleaning and sampling ----------
MIN_WORDS   = 30      # documents shorter than this are flagged "short" and left out of the sample
SAMPLE_N    = 3000    # target documents per community in the analysis sample
MAX_PER_THREAD = 30   # AskMen: at most this many comments from one thread in the sample
SEED        = 42      # makes every "random" step repeatable (see notes)
DROP_DELETED_AUTHORS = True   # also drop content whose author deleted their account (respects deletion)

# ---------- API behaviour ----------
BASE_INTERVAL    = 2.0     # seconds between requests (free volunteer service - be polite)
CHECKPOINT_EVERY = 1000    # save to disk whenever this many new records are waiting
WINDOW_DAYS      = 7       # pull one week at a time; a failing window is automatically cut in half
MIN_WINDOW_SECONDS = 86400 # ...but never smaller than one day

# Usernames are personal data. Only a salted hash is ever written to disk.
# IMPORTANT: use the SAME salt you used in the earlier notebook, or hashes won't match across files.
AUTHOR_SALT = "change-me-to-a-private-string"

# ---------- keywords (applied on your computer, to every community) ----------
# "alone" was dropped on purpose: it matches "leave me alone", "alone time", etc.
KEYWORDS = [   # (group, label, regex)  - word forms included because matching is whole-word
    ("loneliness",      "lonely/loneliness",    r"lonel(?:y|iness)"),
    ("loneliness",      "isolated/isolation",   r"isolat(?:ed|ion)"),
    ("loneliness",      "no friends",           r"no friends"),
    ("loneliness",      "friendless",           r"friendless"),
    ("loneliness",      "disconnected",         r"disconnected"),
    ("mental_health",   "anxiety/anxious",      r"anxi(?:ety|ous)"),
    ("mental_health",   "depression/depressed", r"depress(?:ed|ion)"),
    ("mental_health",   "mental health",        r"mental health"),
    ("mental_health",   "suicide/suicidal",     r"suicid(?:e|al)"),
    ("addiction_anger", "addiction/addicted",   r"addict(?:ed|ion|s)?"),
    ("addiction_anger", "gambling/gambler",     r"gambl(?:ing|ers?)"),
    ("addiction_anger", "anger/angry",          r"ang(?:er|ry)"),
    ("addiction_anger", "rage",                 r"rage"),
]
KW_COMPILED = [(g, l, re.compile(r"\b(?:%s)\b" % p, re.I)) for g, l, p in KEYWORDS]
ANY_KW = re.compile(r"\b(?:%s)\b" % "|".join(p for _, _, p in KEYWORDS), re.I)

def keyword_tags(text):
    """-> (labels matched, groups matched), both as '; '-joined strings."""
    hits = [(g, l) for g, l, rx in KW_COMPILED if rx.search(text)]
    return "; ".join(sorted({l for _, l in hits})), "; ".join(sorted({g for g, _ in hits}))

# ---------- small shared helpers ----------
def to_timestamp(v):
    """Epoch seconds / epoch ms / ISO string -> tz-aware UTC Timestamp (NaT if impossible)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return pd.NaT
    try:
        if isinstance(v, str) and not re.fullmatch(r"-?\d+(\.\d+)?", v.strip()):
            return pd.to_datetime(v, utc=True)
        x = float(v)
        return pd.to_datetime(x, unit="ms" if x > 1e11 else "s", utc=True)
    except Exception:
        return pd.NaT

def to_epoch(v):
    ts = to_timestamp(v)
    return None if pd.isna(ts) else int(ts.timestamp())

def hash_author(name):
    if not isinstance(name, str) or name == "":
        return ""
    if name in ("[deleted]", "AutoModerator"):
        return name
    return hashlib.sha256((AUTHOR_SALT + name.lower()).encode("utf-8")).hexdigest()[:12]

BOT_RE = re.compile(r"(?:[_-]bot|Bot|BOT)$")
def is_bot_name(name):
    return isinstance(name, str) and (name.lower() == "automoderator" or bool(BOT_RE.search(name)))

REMOVED_RE = r"^\s*\[(?:removed|deleted)\]\s*$"      # what Reddit puts where a removed/deleted text used to be

print(f"Window {START_DATE} -> {END_DATE} | TEST_MODE={TEST_MODE} | outputs -> {os.path.abspath(OUTPUT_DIR)}")


Window 2020-01-01 -> 2026-10-22 | TEST_MODE=False | outputs -> C:\Users\vesta\Downloads\project_outputs


In [3]:
# ============================================================
# API CLIENT + RESUMABLE SAVING  (run once; no output expected)
# ============================================================
ARCTIC = "https://arctic-shift.photon-reddit.com"
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "mens-loneliness-research/1.0 (university course project)"
arctic_interval = BASE_INTERVAL
_last_call = 0.0

class ApiStruggling(Exception):
    """Raised when many requests in a row fail - we stop cleanly so nothing is lost."""

_consecutive_failed_pieces = 0
MAX_CONSECUTIVE_FAILED_PIECES = 12

def _pace():
    global _last_call
    wait = arctic_interval - (time.time() - _last_call)
    if wait > 0:
        time.sleep(wait)
    _last_call = time.time()

def arctic_get(endpoint, params, max_retries=5):
    """GET one Arctic Shift page. Returns (data, status); status is 'ok', 'timeout' or an error text."""
    global arctic_interval
    timeouts = 0
    for attempt in range(max_retries):
        _pace()
        try:
            r = SESSION.get(ARCTIC + endpoint, params=params, timeout=120)
        except requests.exceptions.RequestException as e:
            print(f"    connection problem ({type(e).__name__}); retrying...")
            time.sleep(5 * (attempt + 1))
            continue
        if r.status_code == 429:                                   # asked to slow down
            arctic_interval = min(arctic_interval * 1.5, 20)
            reset = r.headers.get("X-RateLimit-Reset")
            try:
                wait = float(reset) + 1 if reset else 15 * 2 ** attempt
            except ValueError:
                wait = 15 * 2 ** attempt
            print(f"    rate limited - waiting {wait:.0f}s (attempt {attempt + 1}/{max_retries})")
            time.sleep(min(wait, 900))
            continue
        if r.status_code >= 500:
            time.sleep(5 * (attempt + 1))
            continue
        try:
            payload = r.json()
        except ValueError:
            return None, f"bad_response_http_{r.status_code}"
        err = payload.get("error") if isinstance(payload, dict) else None
        if r.status_code != 200 or err:
            msg = str(err or f"http_{r.status_code}")
            if "timeout" in msg.lower() or "timed out" in msg.lower():    # server too busy for this request size
                arctic_interval = min(arctic_interval * 1.25, 10)          # slow down a little
                timeouts += 1
                if timeouts <= 1:
                    time.sleep(5)
                    continue                                               # one quick retry
                return None, "timeout"
            return None, msg[:200]
        arctic_interval = max(BASE_INTERVAL, arctic_interval * 0.95)       # slowly speed back up
        return (payload.get("data") if isinstance(payload, dict) else payload), "ok"
    return None, "max_retries_exceeded"

def arctic_fetch_all(endpoint, params, after, before, max_items=20000):
    """Page through everything in [after, before) using the created_utc cursor. Returns (items, status)."""
    items, seen = [], set()
    cursor, page_limit = after, "auto"                 # "auto" = 100-1000 rows per request
    while True:
        q = dict(params, after=cursor, before=before, sort="asc", limit=page_limit)
        data, status = arctic_get(endpoint, q)
        if data is None:
            if page_limit == "auto":                   # big pages can time out -> fall back to 100 per page
                page_limit = 100
                continue
            return items, status
        new = [d for d in data if d.get("id") not in seen]
        if not new:
            break
        seen.update(d.get("id") for d in new)
        items.extend(new)
        if len(items) >= max_items:
            return items, "capped"
        if len(data) < 100:                            # a short page is the last page
            break
        last = max((to_epoch(d.get("created_utc")) or cursor) for d in new)
        if last >= before:
            break
        cursor = last
    return items, "ok"

def fetch_window(endpoint, params, a, b, max_items=20000, min_span=None):
    """Fetch [a, b). If it fails (timeout / too big), cut the window in half and try each half.
    Returns (items, log) where log lists every piece as (status, start, end, n_items)."""
    global _consecutive_failed_pieces
    min_span = MIN_WINDOW_SECONDS if min_span is None else min_span
    items, status = arctic_fetch_all(endpoint, params, a, b, max_items)
    if status == "ok":
        _consecutive_failed_pieces = 0
        return items, [("ok", a, b, len(items))]
    if (b - a) > min_span:                              # discard the partial result and retry in two halves
        mid = (a + b) // 2
        left, l_log = fetch_window(endpoint, params, a, mid, max_items, min_span)
        right, r_log = fetch_window(endpoint, params, mid, b, max_items, min_span)
        return left + right, l_log + r_log
    _consecutive_failed_pieces += 1                     # smallest piece still failing: keep what we got, mark incomplete
    if _consecutive_failed_pieces >= MAX_CONSECUTIVE_FAILED_PIECES:
        raise ApiStruggling("The archive keeps failing right now. Everything collected so far is saved - "
                            "wait a while and re-run the same cell; it will continue where it stopped.")
    return items, [(status, a, b, len(items))]

# ---------- date windows ----------
def split_range(a, b, days=WINDOW_DAYS):
    step, out, x = days * 86400, [], a
    while x < b:
        out.append((x, min(x + step, b)))
        x += step
    return out

def make_windows(start=START_DATE, end=END_DATE, days=WINDOW_DAYS):
    a = int(pd.Timestamp(start, tz="UTC").timestamp())
    b = int((pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1)).timestamp())
    w = split_range(a, b, days)
    return w[-TEST_WEEKS:] if TEST_MODE else w

def month_to_range(label):
    p = pd.Period(label, freq="M")
    return (int(p.start_time.tz_localize("UTC").timestamp()),
            int(p.end_time.tz_localize("UTC").timestamp()) + 1)

# ---------- what every saved row looks like ----------
COLUMNS = ["doc_id", "doc_type", "subreddit", "date", "title", "selftext", "body", "score", "num_comments",
           "post_id", "thread_title", "parent_id", "author_hash", "is_bot", "flair", "permalink", "source"]

POST_FIELDS = "id,author,created_utc,subreddit,title,selftext,score,num_comments,link_flair_text,url,over_18"
COMMENT_FIELDS = "id,author,created_utc,subreddit,body,score,link_id,parent_id"

def post_row(d, sub, source):
    ts = to_timestamp(d.get("created_utc"))
    pid = d.get("id", "")
    s = d.get("subreddit") or sub
    return {"doc_id": pid, "doc_type": "post", "subreddit": s,
            "date": ts.isoformat() if pd.notna(ts) else "",
            "title": d.get("title") or "", "selftext": d.get("selftext") or "", "body": "",
            "score": d.get("score", 0) or 0, "num_comments": d.get("num_comments", 0) or 0,
            "post_id": pid, "thread_title": "", "parent_id": "",
            "author_hash": hash_author(d.get("author")), "is_bot": is_bot_name(d.get("author")),
            "flair": d.get("link_flair_text") or "",
            "permalink": f"https://www.reddit.com/r/{s}/comments/{pid}/", "source": source}

# ---------- resumable saving: "save every 1000, then continue" ----------
class Checkpoint:
    """Collects rows in memory and appends them to a CSV on disk whenever CHECKPOINT_EVERY are waiting.
    A small progress file remembers which units (weeks / threads) are finished, so a re-run skips them."""
    def __init__(self, name):
        self.data_path = os.path.join(CHECK_DIR, f"{name}.csv")
        self.prog_path = os.path.join(CHECK_DIR, f"{name}__progress.csv")
        self.buf, self.pending, self.n_saved, self.done = [], [], 0, set()
        if os.path.exists(self.prog_path):
            p = pd.read_csv(self.prog_path, dtype=str, keep_default_na=False)
            self.done = set(p.loc[p["status"] == "complete", "unit"])

    def add(self, rows, unit, complete, n_failed=0):
        self.buf.extend(rows)
        self.pending.append({"unit": str(unit), "status": "complete" if complete else "incomplete",
                             "n_records": len(rows), "n_failed_pieces": n_failed})
        if len(self.buf) >= CHECKPOINT_EVERY:
            self.flush()

    def flush(self):
        if self.buf:
            df = pd.DataFrame(self.buf).reindex(columns=COLUMNS)
            df.to_csv(self.data_path, mode="a", header=not os.path.exists(self.data_path),
                      index=False, encoding="utf-8")
            self.n_saved += len(self.buf)
            self.buf = []
        if self.pending:
            pd.DataFrame(self.pending).to_csv(self.prog_path, mode="a",
                                              header=not os.path.exists(self.prog_path), index=False)
            self.pending = []

def read_checkpoint(name):
    """All rows saved so far under this name (duplicates from interrupted runs removed)."""
    path = os.path.join(CHECK_DIR, f"{name}.csv")
    if not os.path.exists(path):
        return pd.DataFrame(columns=COLUMNS)
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    return df.drop_duplicates(subset=["doc_type", "doc_id"], keep="last").reset_index(drop=True)

def pull_posts(sub, windows, name, source, keyword_filter=False):
    """Download every post of r/sub window by window, saving as we go.
    keyword_filter=True: download everything but SAVE only posts that contain a keyword."""
    cp = Checkpoint(name)
    todo = [(a, b) for a, b in windows if str(a) not in cp.done]
    print(f"r/{sub} [{name}]: {len(todo)} windows to fetch ({len(windows) - len(todo)} already finished earlier)")
    pulled = kept = incomplete = 0
    try:
        for i, (a, b) in enumerate(todo, 1):
            items, log = fetch_window("/api/posts/search", {"subreddit": sub, "fields": POST_FIELDS}, a, b)
            n_failed = sum(1 for st, *_ in log if st != "ok")
            rows = [post_row(d, sub, source) for d in items]
            pulled += len(rows)
            if keyword_filter:
                rows = [r for r in rows if ANY_KW.search(r["title"] + " " + r["selftext"])]
            kept += len(rows)
            incomplete += n_failed > 0
            cp.add(rows, a, n_failed == 0, n_failed)
            if i % 5 == 0 or i == len(todo):
                print(f"  window {i}/{len(todo)} ({pd.to_datetime(a, unit='s').date()}): "
                      f"{pulled:,} downloaded, {kept:,} kept | windows with gaps so far: {incomplete}")
    except ApiStruggling as e:
        print("\nSTOPPED EARLY:", e)
    except KeyboardInterrupt:
        print("\nInterrupted - saving what we have...")
    finally:
        cp.flush()
    print(f"r/{sub} [{name}] done for now: {cp.n_saved:,} rows saved this run -> {cp.data_path}")
    if incomplete:
        print(f"  {incomplete} windows still have gaps (see progress file). Re-running this cell retries only those.")
    return cp

print("API client ready.")


API client ready.


In [4]:
# ---- PATCH: stop cleanly when the archive can't be reached + save every 5 weeks ----
_conn_fail_streak = 0

def arctic_get(endpoint, params, max_retries=5):
    """Same as before, but gives up cleanly after ~6 connection failures in a row."""
    global arctic_interval, _conn_fail_streak
    timeouts = 0
    for attempt in range(max_retries):
        _pace()
        try:
            r = SESSION.get(ARCTIC + endpoint, params=params, timeout=120)
        except requests.exceptions.RequestException as e:
            _conn_fail_streak += 1
            print(f"    connection problem ({type(e).__name__}); streak {_conn_fail_streak}/6")
            if _conn_fail_streak >= 6:
                _conn_fail_streak = 0
                raise ApiStruggling("Can't reach the archive (check your internet, VPN, and "
                                    "status.arctic-shift.photon-reddit.com). Everything collected so far is "
                                    "saved - fix the connection, then re-run the same cell to continue.")
            time.sleep(5 * (attempt + 1))
            continue
        _conn_fail_streak = 0                      # we got an answer from the server
        if r.status_code == 429:
            arctic_interval = min(arctic_interval * 1.5, 20)
            reset = r.headers.get("X-RateLimit-Reset")
            try:
                wait = float(reset) + 1 if reset else 15 * 2 ** attempt
            except ValueError:
                wait = 15 * 2 ** attempt
            print(f"    rate limited - waiting {wait:.0f}s (attempt {attempt + 1}/{max_retries})")
            time.sleep(min(wait, 900))
            continue
        if r.status_code >= 500:
            time.sleep(5 * (attempt + 1))
            continue
        try:
            payload = r.json()
        except ValueError:
            return None, f"bad_response_http_{r.status_code}"
        err = payload.get("error") if isinstance(payload, dict) else None
        if r.status_code != 200 or err:
            msg = str(err or f"http_{r.status_code}")
            if "timeout" in msg.lower() or "timed out" in msg.lower():
                arctic_interval = min(arctic_interval * 1.25, 10)
                timeouts += 1
                if timeouts <= 1:
                    time.sleep(5)
                    continue
                return None, "timeout"
            return None, msg[:200]
        arctic_interval = max(BASE_INTERVAL, arctic_interval * 0.95)
        return (payload.get("data") if isinstance(payload, dict) else payload), "ok"
    return None, "max_retries_exceeded"

def _checkpoint_add(self, rows, unit, complete, n_failed=0):
    self.buf.extend(rows)
    self.pending.append({"unit": str(unit), "status": "complete" if complete else "incomplete",
                         "n_records": len(rows), "n_failed_pieces": n_failed})
    if len(self.buf) >= CHECKPOINT_EVERY or len(self.pending) >= 5:     # also save every 5 weeks
        self.flush()
Checkpoint.add = _checkpoint_add
print("Patch applied.")

Patch applied.


## Step A — Fill the gaps in r/lonely and r/ForeverAlone
Your old Diagnostics file shows many months marked `partial`: the archive timed out on part of the month, so some weeks are missing. This cell reads that file and re-downloads **only those months**, in 1-week windows (a window that fails is cut in half and retried, down to 1 day). Posts already in your old file are not duplicated (they are matched by post id in Step E).

In [15]:
# ============================================================
# STEP A - re-pull ONLY the incomplete months of r/lonely and r/ForeverAlone
# (reads your old Diagnostics file; complete months are left alone)
# ============================================================
diag = pd.read_csv("C:/Users/vesta/Downloads/project_outputs/reddit_1B_posts_by_popularity__Diagnostics.csv")
complete_status = {"ok", "ok(split)"}
bad = diag[diag["subreddit"].isin(SUBREDDITS_OLD) & ~diag["status"].isin(complete_status)]

for sub in SUBREDDITS_OLD:
    months = sorted(bad.loc[bad["subreddit"] == sub, "month_piece"].unique())
    total = diag.loc[diag["subreddit"] == sub, "month_piece"].nunique()
    print(f"r/{sub}: {len(months)} of {total} months were incomplete in the old run.")
    if TEST_MODE:
        months = months[:1]
        print("  (TEST_MODE: only re-pulling the first incomplete month)")
    windows = []
    for m in months:
        a, b = month_to_range(m)
        windows += split_range(a, b, WINDOW_DAYS)
    if windows:
        pull_posts(sub, windows, name=f"repull_{sub}", source="repull")


r/lonely: 62 of 82 months were incomplete in the old run.
r/lonely [repull_lonely]: 308 windows to fetch (0 already finished earlier)
  window 5/308 (2020-02-29): 4,273 downloaded, 4,273 kept | windows with gaps so far: 0
  window 10/308 (2020-03-29): 8,566 downloaded, 8,566 kept | windows with gaps so far: 0
  window 15/308 (2020-04-29): 12,590 downloaded, 12,590 kept | windows with gaps so far: 0
  window 20/308 (2020-05-29): 16,607 downloaded, 16,607 kept | windows with gaps so far: 0
  window 25/308 (2020-06-29): 20,551 downloaded, 20,551 kept | windows with gaps so far: 0
  window 30/308 (2020-07-29): 25,081 downloaded, 25,081 kept | windows with gaps so far: 0
  window 35/308 (2020-08-29): 29,340 downloaded, 29,340 kept | windows with gaps so far: 0
  window 40/308 (2020-09-29): 33,396 downloaded, 33,396 kept | windows with gaps so far: 0
  window 45/308 (2020-10-29): 37,799 downloaded, 37,799 kept | windows with gaps so far: 0
  window 50/308 (2020-11-29): 42,029 downloaded, 42,

## Step B — r/AskMen posts (keyword list applied on your computer)
The archive's own keyword search kept timing out on r/AskMen (most of the old pieces were `partial`). So this cell downloads AskMen week by week and **keeps only posts containing a keyword** from the list in the settings cell. Nothing else is saved. Keywords are matched as whole words (word forms are listed), case-insensitive.

In [5]:
# ============================================================
# STEP B - r/AskMen POSTS with the new keyword list
# The archive's own keyword search keeps timing out, so we download AskMen week by week and
# apply the keyword filter on your computer. Only matching posts are saved.
# (Long run - leave it going; it saves as it goes and can be re-run to continue.)
# ============================================================
cp_askmen = pull_posts(ASKMEN, make_windows(), name="askmen_posts", source="askmen_new", keyword_filter=True)


r/AskMen [askmen_posts]: 47 windows to fetch (309 already finished earlier)
  window 5/47 (2025-12-31): 10,337 downloaded, 392 kept | windows with gaps so far: 0
  window 10/47 (2026-02-04): 21,034 downloaded, 777 kept | windows with gaps so far: 0
    connection problem (ConnectionError); streak 1/6
  window 15/47 (2026-03-11): 31,125 downloaded, 1,148 kept | windows with gaps so far: 0
  window 20/47 (2026-04-15): 40,900 downloaded, 1,489 kept | windows with gaps so far: 0
  window 25/47 (2026-05-20): 52,347 downloaded, 1,853 kept | windows with gaps so far: 0
  window 30/47 (2026-06-24): 64,066 downloaded, 2,274 kept | windows with gaps so far: 0
  window 35/47 (2026-07-29): 74,130 downloaded, 2,618 kept | windows with gaps so far: 0
  window 40/47 (2026-09-02): 83,199 downloaded, 2,935 kept | windows with gaps so far: 0
  window 45/47 (2026-10-07): 91,954 downloaded, 3,243 kept | windows with gaps so far: 0
  window 47/47 (2026-10-21): 93,754 downloaded, 3,305 kept | windows with g

## Step C — r/exredpill posts (all posts)

In [5]:
# ============================================================
# STEP C - r/exredpill POSTS (all posts; separate cell on purpose)
# ============================================================
cp_exredpill = pull_posts(EXREDPILL, make_windows(), name="exredpill_posts", source="exredpill_new")


r/exredpill [exredpill_posts]: 356 windows to fetch (0 already finished earlier)
  window 5/356 (2020-01-29): 80 downloaded, 80 kept | windows with gaps so far: 0
  window 10/356 (2020-03-04): 156 downloaded, 156 kept | windows with gaps so far: 0
  window 15/356 (2020-04-08): 252 downloaded, 252 kept | windows with gaps so far: 0
  window 20/356 (2020-05-13): 345 downloaded, 345 kept | windows with gaps so far: 0
  window 25/356 (2020-06-17): 467 downloaded, 467 kept | windows with gaps so far: 0
  window 30/356 (2020-07-22): 613 downloaded, 613 kept | windows with gaps so far: 0
  window 35/356 (2020-08-26): 790 downloaded, 790 kept | windows with gaps so far: 0
  window 40/356 (2020-09-30): 1,010 downloaded, 1,010 kept | windows with gaps so far: 0
  window 45/356 (2020-11-04): 1,230 downloaded, 1,230 kept | windows with gaps so far: 0
  window 50/356 (2020-12-09): 1,453 downloaded, 1,453 kept | windows with gaps so far: 0
  window 55/356 (2021-01-13): 1,716 downloaded, 1,716 kept |

## Step D — r/AskMen comments
In r/AskMen the post is a *question* and the men's own accounts are in the *replies*. For each year, this cell randomly picks up to 50 of the keyword-matched threads that have at least 20 comments (the same random pick every time, thanks to the seed) and downloads their **top-level comments** (direct replies to the post, not replies to replies). The chosen threads are saved to `askmen_thread_selection.csv`.

In [6]:
# ============================================================
# STEP D - r/AskMen COMMENTS (top-level replies) for a random set of keyword-matched threads
# ============================================================
def select_threads():
    p = read_checkpoint("askmen_posts")
    if p.empty:
        raise RuntimeError("No AskMen posts yet - run Step B first.")
    p["num_comments"] = pd.to_numeric(p["num_comments"], errors="coerce").fillna(0)
    p["date"] = pd.to_datetime(p["date"], utc=True, errors="coerce")
    p["year"] = p["date"].dt.year
    removed = (p["selftext"].str.match(REMOVED_RE, case=False) | p["title"].str.match(REMOVED_RE, case=False))
    p = p[~removed & (p["author_hash"] != "[deleted]") & (p["author_hash"] != "AutoModerator")
          & (p["num_comments"] >= MIN_COMMENTS_IN_THREAD)]
    picks = [g.sample(n=min(len(g), THREADS_PER_YEAR), random_state=SEED) for _, g in p.groupby("year")]
    chosen = pd.concat(picks).sort_values("date") if picks else p.iloc[0:0]
    return chosen.head(3) if TEST_MODE else chosen

threads = select_threads()
threads.to_csv(os.path.join(OUTPUT_DIR, f"askmen_thread_selection{TAG}.csv"), index=False, encoding="utf-8-sig")
print(f"{len(threads)} threads selected ({threads['year'].nunique()} years). Per year:")
print(threads.groupby("year").size().to_string())

# --- the API wants the post id with or without the 't3_' prefix; test once which one works ---
LINK_PREFIX = None
def detect_link_prefix(post_id, post_epoch):
    for prefix in ["", "t3_"]:
        items, status = arctic_fetch_all("/api/comments/search",
                                         {"link_id": prefix + post_id, "fields": COMMENT_FIELDS},
                                         post_epoch - 1, post_epoch + 30 * 86400, max_items=5)
        if items:
            return prefix
    return None

def comment_row(c, post_id, thread_title):
    ts = to_timestamp(c.get("created_utc"))
    return {"doc_id": c.get("id", ""), "doc_type": "comment", "subreddit": c.get("subreddit") or ASKMEN,
            "date": ts.isoformat() if pd.notna(ts) else "", "title": "", "selftext": "",
            "body": c.get("body") or "", "score": c.get("score", 0) or 0, "num_comments": 0,
            "post_id": post_id, "thread_title": thread_title, "parent_id": c.get("parent_id", ""),
            "author_hash": hash_author(c.get("author")), "is_bot": is_bot_name(c.get("author")),
            "flair": "", "permalink": "", "source": "askmen_comments"}

cp_comments = Checkpoint("askmen_comments")
todo = threads[~threads["doc_id"].isin(cp_comments.done)]
print(f"\n{len(todo)} threads to fetch ({len(threads) - len(todo)} already finished earlier)")
try:
    for i, (_, t) in enumerate(todo.iterrows(), 1):
        post_epoch = int(t["date"].timestamp())
        if LINK_PREFIX is None:
            LINK_PREFIX = detect_link_prefix(t["doc_id"], post_epoch)
            if LINK_PREFIX is None:
                print(f"No comments came back for thread {t['doc_id']} in either id format - skipping it for now.")
                continue
            print(f"(comment lookup format found: link_id prefix = '{LINK_PREFIX}')")
        items, status = arctic_fetch_all("/api/comments/search",
                                         {"link_id": LINK_PREFIX + t["doc_id"], "fields": COMMENT_FIELDS},
                                         post_epoch - 1, int(time.time()) + 86400, max_items=MAX_RAW_COMMENTS)
        top = [c for c in items if str(c.get("parent_id", "")).startswith("t3_")][:MAX_TOPLEVEL]
        rows = [comment_row(c, t["doc_id"], t["title"]) for c in top]
        ok = status in ("ok", "capped")
        cp_comments.add(rows, t["doc_id"], ok, 0 if ok else 1)
        if i % 10 == 0 or i == len(todo):
            print(f"  thread {i}/{len(todo)}: {len(items)} comments downloaded, {len(rows)} top-level kept"
                  + ("" if ok else f"  [incomplete: {status}]"))
except KeyboardInterrupt:
    print("\nInterrupted - saving what we have...")
finally:
    cp_comments.flush()
print("Comments saved to", cp_comments.data_path)


700 threads selected (7 years). Per year:
year
2020    100
2021    100
2022    100
2023    100
2024    100
2025    100
2026    100

700 threads to fetch (0 already finished earlier)
(comment lookup format found: link_id prefix = '')
  thread 10/700: 31 comments downloaded, 8 top-level kept
  thread 20/700: 29 comments downloaded, 12 top-level kept
  thread 30/700: 189 comments downloaded, 79 top-level kept
  thread 40/700: 23 comments downloaded, 18 top-level kept
  thread 50/700: 668 comments downloaded, 186 top-level kept
  thread 60/700: 36 comments downloaded, 10 top-level kept
  thread 70/700: 171 comments downloaded, 58 top-level kept
  thread 80/700: 43 comments downloaded, 28 top-level kept
  thread 90/700: 136 comments downloaded, 84 top-level kept
  thread 100/700: 32 comments downloaded, 22 top-level kept
  thread 110/700: 43 comments downloaded, 31 top-level kept
  thread 120/700: 88 comments downloaded, 45 top-level kept
  thread 130/700: 25 comments downloaded, 17 top-lev

## Step E — Combine
Takes the old file, **drops r/AskMen (partial run) and r/MGTOW (banned)**, adds the re-pulled months, the new AskMen posts and comments, and r/exredpill, then removes duplicate ids (newer pulls replace older ones).

In [9]:
# ============================================================
# STEP E - COMBINE: old r/lonely + r/ForeverAlone (AskMen + MGTOW removed)
#                    + re-pulled months + new AskMen posts/comments + new r/exredpill
# ============================================================
old = pd.read_csv("C:/Users/vesta/Downloads/project_outputs/reddit_1B_posts_by_popularity__All_matching_posts.csv", dtype=str, keep_default_na=False, encoding="utf-8-sig")
print("Old file rows per subreddit:")
print(old["subreddit"].value_counts().to_string())

old_kept = old[~old["subreddit"].isin(DROP_FROM_OLD)].copy()
print(f"\nRemoved {len(old) - len(old_kept):,} old rows from {DROP_FROM_OLD}; kept {len(old_kept):,}.")
old_kept.to_csv(os.path.join(OUTPUT_DIR, "old_1B_without_AskMen_MGTOW.csv"), index=False, encoding="utf-8-sig")

def old_to_schema(df):
    ah = df["author_hash"] if "author_hash" in df else pd.Series("", index=df.index)
    col = lambda c: df[c] if c in df else pd.Series("", index=df.index)
    out = pd.DataFrame({
        "doc_id": df["post_id"], "doc_type": "post", "subreddit": df["subreddit"], "date": df["date"],
        "title": col("title"), "selftext": col("selftext"), "body": "",
        "score": col("score"), "num_comments": col("num_comments"), "post_id": df["post_id"],
        "thread_title": "", "parent_id": "", "author_hash": ah, "is_bot": ah.eq("AutoModerator"),
        "flair": col("flair"), "permalink": col("permalink"), "source": "old_pull"})
    return out

parts = [old_to_schema(old_kept)]
for name in [f"repull_{s}" for s in SUBREDDITS_OLD] + ["askmen_posts", "exredpill_posts", "askmen_comments"]:
    df = read_checkpoint(name)
    print(f"{name}: {len(df):,} rows")
    if len(df):
        parts.append(df)

for p in parts:                                   # one date format, always UTC
    p["date"] = pd.to_datetime(p["date"], utc=True, errors="coerce")

combined = pd.concat(parts, ignore_index=True)
# make sure each community is spelled one way, whatever capitalisation the archive returned
canon = {n.lower(): n for n in SUBREDDITS_OLD + [ASKMEN, EXREDPILL]}
combined["subreddit"] = combined["subreddit"].map(lambda x: canon.get(str(x).lower(), x))
before = len(combined)
combined = combined.drop_duplicates(subset=["doc_type", "doc_id"], keep="last")   # newer pulls win over older ones
print(f"\nCombined: {before:,} rows -> {len(combined):,} after removing duplicate ids")
print(combined.groupby(["subreddit", "doc_type"]).size().to_string())
combined.to_csv(os.path.join(OUTPUT_DIR, f"combined_raw{TAG}.csv"), index=False, encoding="utf-8-sig")


Old file rows per subreddit:
subreddit
lonely          367568
ForeverAlone     93903
AskMen            4590

Removed 4,590 old rows from ['AskMen', 'MGTOW']; kept 461,471.
repull_lonely: 385,534 rows
repull_ForeverAlone: 23,470 rows
askmen_posts: 28,045 rows
exredpill_posts: 11,742 rows
askmen_comments: 36,662 rows

Combined: 946,924 rows -> 656,505 after removing duplicate ids
subreddit     doc_type
AskMen        comment      36662
              post         28045
ForeverAlone  post         97357
exredpill     post         11742
lonely        post        482699


## Step F — Clean and tag
**Cleaning** removes: duplicate ids, `[removed]`/`[deleted]` text (any capitalisation, extra spaces), posts whose author deleted their account (switch in settings), AutoModerator and bots, empty text, and exact-duplicate text. Short documents (under `MIN_WORDS`) are **flagged, not deleted**.
**Tagging** adds labels and removes nothing: which keywords a document contains, a gender tag if the title has one like `(24F)`, `[M25]` or `24f` (upper and lower case), and a popularity rank within each community and year.
The **F-tag report** compares the old naive `"F "` search with real tags, so you can see how much the naive search was catching ordinary words (like "IF " or "OF "). The loose pattern (e.g. `24 f`) is less reliable than the bracketed one — `20 m` can mean metres.

In [10]:
# ============================================================
# STEP F - CLEAN + TAG   (nothing here touches the internet)
# Cleaning = remove things that should not be analysed.   Tagging = add labels; removes nothing.
# ============================================================
raw = combined                      # (re-run Step E first if you ever re-run this cell after an error)
for c in ["score", "num_comments"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce").fillna(0).astype(int)
for c in ["title", "selftext", "body", "thread_title", "author_hash"]:
    raw[c] = raw[c].fillna("").astype(str)
raw["is_bot"] = raw["is_bot"].astype(str).str.lower().eq("true")

snaps = {}
def snap(name, df):
    s = df.groupby(["subreddit", "doc_type"]).size()
    s.index = [f"{a}/{b}" for a, b in s.index]
    snaps[name] = s

# ---------- cleaning ----------
snap("0. combined file", raw)

df = raw.drop_duplicates(subset=["doc_type", "doc_id"])
snap("1. duplicate ids removed", df)

removed = (df["title"].str.match(REMOVED_RE, case=False) | df["selftext"].str.match(REMOVED_RE, case=False)
           | df["body"].str.match(REMOVED_RE, case=False))            # "[removed]" / "[deleted]", any case, extra spaces ok
df = df[~removed]
snap("2. removed/deleted by Reddit, mods or user", df)

if DROP_DELETED_AUTHORS:
    df = df[df["author_hash"] != "[deleted]"]
snap("3. author deleted their account", df)

df = df[~(df["is_bot"] | df["author_hash"].eq("AutoModerator"))]
snap("4. AutoModerator / bots", df)

def normalize_text(s):
    s = html.unescape(s)                                           # &amp; -> &, &#x200B; -> invisible char
    s = re.sub(r"\[([^\]]+)\]\(https?://[^)]+\)", r"\1", s)        # markdown link -> keep the link text
    s = re.sub(r"https?://\S+|www\.\S+", " ", s)                   # bare links
    s = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", s)         # invisible characters
    return re.sub(r"\s+", " ", s).strip()

is_post = df["doc_type"].eq("post")
df = df.assign(text=np.where(is_post, df["title"] + " " + df["selftext"], df["body"]))
df["text"] = df["text"].map(normalize_text)
df["n_words"] = df["text"].str.split().str.len().fillna(0).astype(int)
df = df[df["n_words"] > 0]
snap("5. empty text", df)

dup = df.duplicated(subset=["subreddit", "doc_type", "text"], keep="first") & (df["n_words"] >= 10)
df = df[~dup]                                                      # same text posted twice (spam, copy-paste)
snap("6. exact-duplicate text (10+ words)", df)

df = df.copy()
df["is_short"] = df["n_words"] < MIN_WORDS                         # flagged, NOT removed (sampling skips them)
df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
df["year"] = df["date"].dt.year.fillna(0).astype(int)

cleaning_log = pd.DataFrame(snaps).T.fillna(0).astype(int)      # one row per step, one column per community
print("CLEANING LOG - documents left after each step")
print(cleaning_log.to_string())

print(f"\nShare of documents with at least N words (MIN_WORDS is {MIN_WORDS}):")
wc = pd.DataFrame({f">= {t} words (%)": (df["n_words"] >= t).groupby([df["subreddit"], df["doc_type"]]).mean().mul(100).round(1)
                   for t in (20, 30, 50)})
print(wc.to_string())

# ---------- tagging (adds labels, removes nothing) ----------
# 1) keywords - the SAME list for every community
def tag_kw(t):
    return ("", "") if not ANY_KW.search(t) else keyword_tags(t)
tags = df["text"].map(tag_kw)
df["kw_labels"] = tags.map(lambda x: x[0])
df["kw_groups"] = tags.map(lambda x: x[1])

# 2) gender tags such as (24F), [M25], 23f in post titles - upper AND lower case
BRACKET_TAG = re.compile(r"[\(\[\{]\s*(?:(\d{2})\s*([mf])|([mf])\s*(\d{2}))\s*[\)\]\}]", re.I)
LOOSE_TAG   = re.compile(r"(?<![\w.'\u2019])(?:(\d{2})\s?([mf])|([mf])\s?(\d{2}))(?!\w)", re.I)   # not after an apostrophe: "I'm 24" is not an M tag
def gender_tag(title):
    for rx, kind in ((BRACKET_TAG, "bracket"), (LOOSE_TAG, "loose")):
        for m in rx.finditer(title):
            age = int(m.group(1) or m.group(4))
            if 13 <= age <= 90:
                return (m.group(2) or m.group(3)).upper(), kind
    return "", ""
gt = df["title"].map(gender_tag)
df["gender_tag"] = gt.map(lambda x: x[0])
df["gender_tag_type"] = gt.map(lambda x: x[1])
df.loc[~df["doc_type"].eq("post"), ["gender_tag", "gender_tag_type"]] = ""      # only posts have titles

# 3) popularity rank inside each community / year (for reading examples; NOT used for sampling)
df["rank_in_year"] = (df.groupby(["subreddit", "doc_type", "year"])["score"]
                        .rank(method="first", ascending=False).astype(int))

# ---------- F-tag report: how many are real tags vs. the old naive "F " search? ----------
posts = df[df["doc_type"].eq("post")]
by_sub = posts["subreddit"]
cnt = lambda m: m.groupby(by_sub).sum().astype(int)
report = pd.DataFrame({
    "posts": posts.groupby("subreddit").size(),
    'naive "F " (capital F + space)': cnt(posts["text"].str.contains("F ", regex=False)),
    'naive "f " (lowercase)':         cnt(posts["text"].str.contains("f ", regex=False)),
    "real F tag, bracketed e.g. (24F)": cnt(posts["gender_tag"].eq("F") & posts["gender_tag_type"].eq("bracket")),
    "real F tag, loose e.g. 24f":       cnt(posts["gender_tag"].eq("F") & posts["gender_tag_type"].eq("loose")),
    "M tag (any)":                      cnt(posts["gender_tag"].eq("M")),
}).fillna(0).astype(int)
report["% of posts with an F tag"] = ((report.iloc[:, 3] + report.iloc[:, 4]) / report["posts"] * 100).round(1)
print("\nF-TAG REPORT (posts only)")
print(report.to_string())

naive_only = posts[posts["text"].str.contains("F ", regex=False) & ~posts["gender_tag"].eq("F")]
print(f"\n{len(naive_only):,} posts match the naive 'F ' search but have NO real tag. Examples of what it caught:")
for t in naive_only["text"].sample(n=min(8, len(naive_only)), random_state=SEED):
    i = t.find("F ")
    print("   ...", t[max(0, i - 25): i + 25].replace("\n", " "), "...")

# ---------- save ----------
keep_cols = ["doc_id", "doc_type", "subreddit", "date", "year", "title", "text", "n_words", "is_short",
             "kw_labels", "kw_groups", "gender_tag", "gender_tag_type", "score", "num_comments", "rank_in_year",
             "post_id", "thread_title", "parent_id", "author_hash", "flair", "permalink", "source"]
cleaned = df[keep_cols].reset_index(drop=True)
cleaned.to_csv(os.path.join(OUTPUT_DIR, f"cleaned_all{TAG}.csv"), index=False, encoding="utf-8-sig")
cleaning_log.to_csv(os.path.join(OUTPUT_DIR, f"cleaning_log{TAG}.csv"), encoding="utf-8-sig")
report.to_csv(os.path.join(OUTPUT_DIR, f"f_tag_report{TAG}.csv"), encoding="utf-8-sig")
print(f"\nSaved cleaned_all{TAG}.csv ({len(cleaned):,} rows), cleaning_log{TAG}.csv, f_tag_report{TAG}.csv")


CLEANING LOG - documents left after each step
                                            AskMen/comment  AskMen/post  ForeverAlone/post  exredpill/post  lonely/post
0. combined file                                     36662        28045              97357           11742       482699
1. duplicate ids removed                             36662        28045              97357           11742       482699
2. removed/deleted by Reddit, mods or user           33792        14328              50746            7293       332795
3. author deleted their account                      33128        14127              49194            6936       325946
4. AutoModerator / bots                              32950        14121              49194            6936       325818
5. empty text                                        32930        14121              49191            6935       325817
6. exact-duplicate text (10+ words)                  32880        14041              49023            6903       3

## Step G — Equal random sample for the topic model
For each community this keeps a **copy** of an equal number of randomly chosen documents: r/lonely, r/ForeverAlone and r/exredpill **posts**; r/AskMen **comments** (at most `MAX_PER_THREAD` from any one thread). Short documents are skipped and only English documents are kept. `n` is `SAMPLE_N`, or the size of the smallest community if that is smaller. Nothing is deleted from `cleaned_all.csv`.

In [11]:
# ============================================================
# STEP G - SAMPLE: an equal-size random sample per community for the topic model
# (a COPY - nothing is deleted from cleaned_all)
# ============================================================
from langdetect import detect, DetectorFactory
DetectorFactory.seed = SEED                      # makes language detection repeatable too

def is_english(text):
    try:
        return detect(text[:1000]) == "en"
    except Exception:
        return False

COMMUNITIES = ["lonely", "ForeverAlone", ASKMEN, EXREDPILL]

def eligible_pool(d, sub):
    if sub == ASKMEN:                            # AskMen: the men's own words are in the COMMENTS
        p = d[(d["subreddit"] == sub) & d["doc_type"].eq("comment")]
    else:                                        # the other communities: the POSTS are the personal accounts
        p = d[(d["subreddit"] == sub) & d["doc_type"].eq("post")]
    p = p[~p["is_short"]]
    p = p.sample(frac=1, random_state=SEED)      # shuffle (repeatable because of the seed)
    if sub == ASKMEN:                            # no thread may dominate the sample
        p = p[p.groupby("post_id").cumcount() < MAX_PER_THREAD]
    return p

def draw_english(pool, n, chunk=500):
    """Walk down the shuffled pool, keeping English documents until we have n."""
    kept, have = [], 0
    for i in range(0, len(pool), chunk):
        part = pool.iloc[i:i + chunk]
        part = part[part["text"].map(is_english)]
        kept.append(part)
        have += len(part)
        if have >= n:
            break
    return (pd.concat(kept) if kept else pool.iloc[0:0]).head(n)

drawn = {}
for sub in COMMUNITIES:
    pool = eligible_pool(cleaned, sub)
    if pool.empty:
        print(f"WARNING: r/{sub} has no eligible documents yet - skipped.")
        continue
    drawn[sub] = draw_english(pool, SAMPLE_N)
    print(f"r/{sub}: pool of {len(pool):,} eligible documents -> {len(drawn[sub]):,} English documents drawn")

if not drawn:
    raise RuntimeError("Nothing to sample - run Steps A-F first.")
n_final = min(len(v) for v in drawn.values())
sample = pd.concat([v.head(n_final) for v in drawn.values()], ignore_index=True)
sample["lang"] = "en"
print(f"\nEqual sample: n = {n_final:,} per community ({len(sample):,} documents in total)")

sample.to_csv(os.path.join(OUTPUT_DIR, f"analysis_sample{TAG}.csv"), index=False, encoding="utf-8-sig")
no_f = sample[~sample["gender_tag"].eq("F")]
no_f.to_csv(os.path.join(OUTPUT_DIR, f"analysis_sample_no_F_tags{TAG}.csv"), index=False, encoding="utf-8-sig")
print(f"Sensitivity file (F-tagged posts removed): {len(no_f):,} of {len(sample):,} documents kept")

summary = sample.groupby("subreddit").agg(
    documents=("doc_id", "size"), doc_type=("doc_type", "first"), median_words=("n_words", "median"),
    share_with_keyword=("kw_groups", lambda s: round((s != "").mean() * 100, 1)),
    F_tagged=("gender_tag", lambda s: int((s == "F").sum())), first_year=("year", "min"), last_year=("year", "max"))
print("\n", summary.to_string())


r/lonely: pool of 257,777 eligible documents -> 3,000 English documents drawn
r/ForeverAlone: pool of 39,709 eligible documents -> 3,000 English documents drawn
r/AskMen: pool of 10,236 eligible documents -> 3,000 English documents drawn
r/exredpill: pool of 5,416 eligible documents -> 3,000 English documents drawn

Equal sample: n = 3,000 per community (12,000 documents in total)
Sensitivity file (F-tagged posts removed): 11,909 of 12,000 documents kept

               documents doc_type  median_words  share_with_keyword  F_tagged  first_year  last_year
subreddit                                                                                          
AskMen             3000  comment          65.0                22.3         0        2020       2026
ForeverAlone       3000     post         134.0                33.0        15        2020       2026
exredpill          3000     post         161.0                18.4        11        2020       2026
lonely             3000     post       

## Notes for the write-up
- **Source:** Arctic Shift, a free volunteer archive of public Reddit data with no uptime guarantee. Confirm with your instructor that this is acceptable for the course and for publication (Reddit's Responsible Builder Policy says research should use the Reddit for Researchers program).
- **Gaps:** about 15% of r/lonely and 3% of r/ForeverAlone was missing from the first pull (estimate from the Diagnostics file). Step A re-pulls those months; report the final number of weeks still incomplete (see the `__progress.csv` files).
- **Who is speaking:** Reddit has no verified gender or location. Gender tags exist only on a subset of posts; use `analysis_sample_no_F_tags.csv` as a sensitivity check. Posts cannot be assigned to Canada.
- **r/AskMen** documents are comments chosen from keyword-matched threads; the other communities' documents are posts that were not keyword-filtered. State this difference when comparing communities, and check topics are not just splitting by document type.
- **Seed:** `SEED = 42` makes every random step repeatable (the thread picks, the shuffles and the language detector).
- **Privacy:** only salted hashes of usernames are saved. Do not quote posts directly in the paper; paraphrase.

In [4]:
import glob
print("Looking in:", os.path.abspath(CHECK_DIR))
for f in sorted(glob.glob(os.path.join(CHECK_DIR, "*"))):
    print(f"  {os.path.basename(f)}  ({os.path.getsize(f):,} bytes)")

Looking in: C:\Users\vesta\Downloads\project_outputs\checkpoints
  askmen_posts.csv  (5,743,930 bytes)
  askmen_posts__progress.csv  (4,986 bytes)
  repull_ForeverAlone.csv  (13,452,377 bytes)
  repull_ForeverAlone__progress.csv  (2,437 bytes)
  repull_lonely.csv  (251,028,218 bytes)
  repull_lonely__progress.csv  (8,563 bytes)


In [4]:
   r = requests.get("https://arctic-shift.photon-reddit.com/api/posts/search",
                    params={"subreddit": "lonely", "limit": 1}, timeout=30)
   print(r.status_code, r.text[:150])

200 {"data":[{"all_awardings":[],"allow_live_comments":false,"approved_at_utc":null,"approved_by":null,"archived":false,"author":"Acceptable-Year-5302","a


In [5]:
cp = Checkpoint("askmen_posts")
todo = [(a, b) for a, b in make_windows() if str(a) not in cp.done]
print(len(todo), "windows left. First few start dates:", [str(pd.to_datetime(a, unit="s", utc=True).date()) for a, _ in todo[:6]])
a, b = todo[0]
data, status = arctic_get("/api/posts/search", {"subreddit": "AskMen", "after": a, "before": a + 86400,
                                                "limit": 100, "sort": "asc", "fields": POST_FIELDS})
print("1-day test of the first stuck week ->", status, "| rows:", None if data is None else len(data))

47 windows left. First few start dates: ['2025-12-03', '2025-12-10', '2025-12-17', '2025-12-24', '2025-12-31', '2026-01-07']
1-day test of the first stuck week -> max_retries_exceeded | rows: None


In [7]:
cp = Checkpoint("askmen_posts")
todo = [(a, b) for a, b in make_windows() if str(a) not in cp.done]
a, b = todo[0][0], todo[0][0] + 86400
c = int(pd.Timestamp("2024-06-03", tz="UTC").timestamp())    # a day that already worked

def raw_test(label, sub, start, end):
    try:
        r = requests.get(ARCTIC + "/api/posts/search",
                         params={"subreddit": sub, "after": start, "before": end, "limit": 10, "fields": "id"},
                         timeout=60)
        print(f"{label:36} HTTP {r.status_code} | {r.text[:100]!r}")
    except Exception as e:
        print(f"{label:36} {type(e).__name__}")

raw_test("AskMen, first stuck day", "AskMen", a, b)
raw_test("AskMen, a day in 2024 (control)", "AskMen", c, c + 86400)
raw_test("lonely, first stuck day", "lonely", a, b)

AskMen, first stuck day              HTTP 500 | '{"data":null,"error":"Internal server error"}'
AskMen, a day in 2024 (control)      HTTP 500 | '{"data":null,"error":"Internal server error"}'
lonely, first stuck day              HTTP 500 | '{"data":null,"error":"Internal server error"}'
